# Supervised Learning

In [35]:
import sys
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.model_selection import LeaveOneOut, train_test_split
from sklearn.preprocessing import StandardScaler

sys.path.append("..")
from src.model import StellarCollapseMLP


In [36]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")


Using device: mps


In [37]:
RANDOM_SEED = 42

## Data split

`X` contains the pre-collapse properties and `y` contains `target_score`. The test set holds 25% of the progenitors. Stratification uses the scores observed in Gold.

In [38]:
gold = pd.read_csv("../data/gold/progenitor_gold.csv")
X = gold.drop(columns=["model", "target_score"])
y = gold["target_score"]

print(f"Progenitors: {len(gold)}")
print(f"Input features: {X.shape[1]}")


Progenitors: 187
Input features: 44


In [39]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_SEED
)

print(f"Train: {len(X_train)} progenitors")
print(f"Test: {len(X_test)} progenitors")

score_counts = pd.DataFrame({
    "all": y.value_counts().sort_index(),
    "train": y_train.value_counts().sort_index(),
    "test": y_test.value_counts().sort_index(),
})
display(score_counts)


Train: 140 progenitors
Test: 47 progenitors


,all,train,test
target_score,,,
0.0,74,56,18
0.2,8,6,2
0.5,20,15,5
0.7,3,2,1
0.8,7,5,2
1.0,75,56,19


## Leave-One-Out validation

Each fold leaves one training progenitor for validation. The test set stays separate. The scaler and network will be fitted again in every fold.

In [40]:
loo = LeaveOneOut()
print(f"Leave-One-Out folds: {loo.get_n_splits(X_train)}")

Leave-One-Out folds: 140
